In [4]:
import pandas as pd
import json
import os
import sys

In [8]:
def load_experiment_results(exp_dir):
    rows = []

    # iterate over models (subfolders)
    for model_name in os.listdir(exp_dir):
        model_path = os.path.join(exp_dir, model_name)

        if not os.path.isdir(model_path):
            continue

        # iterate over json files
        for file in os.listdir(model_path):
            if not file.endswith(".json"):
                continue

            file_path = os.path.join(model_path, file)

            with open(file_path, "r") as f:
                data = json.load(f)

            # extract info
            row = {
                "model": data.get("model"),
                "feature_set": data.get("feature_set"),
                "test_set": data.get("test_set"),
            }

            # extract overall metrics
            overall = data.get("result", {}).get("overall", {})

            for k, v in overall.items():
                row[k] = v

            rows.append(row)

    return pd.DataFrame(rows)

In [35]:
def load_stress_light_results(exp_dir, task_type="composition"):
    rows = []

    for model_name in os.listdir(exp_dir):
        model_path = os.path.join(exp_dir, model_name)

        if not os.path.isdir(model_path):
            continue

        for file in os.listdir(model_path):
            if not file.endswith(".json"):
                continue

            with open(os.path.join(model_path, file), "r") as f:
                data = json.load(f)

            overall = data.get("stress_result_light", {}).get("overall", {})

            row = {
                "model": data.get("model"),
                "feature_set": data.get("feature_set"),
                "test_set": data.get("test_set"),
            }

            if task_type == "composition":
                row.update({
                    "mae_macro": overall.get("mae_macro"),
                    "rmse_macro": overall.get("rmse_macro"),
                    "r2_macro": overall.get("r2_macro"),
                    "kl": overall.get("kl"),
                })

            elif task_type == "change":
                row.update({
                    "mcr": overall.get("mcr"),
                    "fcr": overall.get("fcr"),
                    "direction_accuracy": overall.get("direction_accuracy"),
                })

            else:
                raise ValueError(f"Unknown task_type: {task_type}")

            rows.append(row)

    return pd.DataFrame(rows)


def load_stress_strong_results(exp_dir, task_type="composition"):
    rows = []

    for model_name in os.listdir(exp_dir):
        model_path = os.path.join(exp_dir, model_name)

        if not os.path.isdir(model_path):
            continue

        for file in os.listdir(model_path):
            if not file.endswith(".json"):
                continue

            with open(os.path.join(model_path, file), "r") as f:
                data = json.load(f)

            overall = data.get("stress_result_strong", {}).get("overall", {})

            row = {
                "model": data.get("model"),
                "feature_set": data.get("feature_set"),
                "test_set": data.get("test_set"),
            }

            if task_type == "composition":
                row.update({
                    "mae_macro": overall.get("mae_macro"),
                    "rmse_macro": overall.get("rmse_macro"),
                    "r2_macro": overall.get("r2_macro"),
                    "kl": overall.get("kl"),
                })

            elif task_type == "change":
                row.update({
                    "mcr": overall.get("mcr"),
                    "fcr": overall.get("fcr"),
                    "direction_accuracy": overall.get("direction_accuracy"),
                })

            else:
                raise ValueError(f"Unknown task_type: {task_type}")

            rows.append(row)

    return pd.DataFrame(rows)

In [36]:
composition_df = load_experiment_results("../results_repro/exp_2026-03-23_20-44-10-composition")
change_df = load_experiment_results("../results_repro/exp_2026-03-23_21-01-14-change")

composition_light_stress_df = load_stress_light_results("../results_repro/exp_2026-03-23_20-44-10-composition", "composition")
change_light_stress_df = load_stress_light_results("../results_repro/exp_2026-03-23_21-01-14-change", "change")

composition_strong_stress_df = load_stress_strong_results("../results_repro/exp_2026-03-23_20-44-10-composition", "composition")
change_strong_stress_df = load_stress_strong_results("../results_repro/exp_2026-03-23_21-01-14-change", "change")

In [38]:
def merge_stress_results(df_clean, df_light, df_strong, task_type="composition"):
    if task_type == "composition":
        rename_map_light = {
            "mae_macro": "stress_light_mae_macro",
            "rmse_macro": "stress_light_rmse_macro",
            "r2_macro": "stress_light_r2_macro",
            "kl": "stress_light_kl",
        }

        rename_map_strong = {
            "mae_macro": "stress_strong_mae_macro",
            "rmse_macro": "stress_strong_rmse_macro",
            "r2_macro": "stress_strong_r2_macro",
            "kl": "stress_strong_kl",
        }

    elif task_type == "change":
        rename_map_light = {
            "mcr": "stress_light_mcr",
            "fcr": "stress_light_fcr",
            "direction_accuracy": "stress_light_direction_accuracy",
        }

        rename_map_strong = {
            "mcr": "stress_strong_mcr",
            "fcr": "stress_strong_fcr",
            "direction_accuracy": "stress_strong_direction_accuracy",
        }

    else:
        raise ValueError(f"Unknown task_type: {task_type}")

    # --- rename ---
    light = df_light.rename(columns=rename_map_light)
    strong = df_strong.rename(columns=rename_map_strong)

    # --- merge ---
    df = df_clean.merge(
        light,
        on=["model", "feature_set", "test_set"],
        how="left",
    )

    df = df.merge(
        strong,
        on=["model", "feature_set", "test_set"],
        how="left",
    )

    return df

In [39]:
stacked_composition = merge_stress_results(composition_df, composition_light_stress_df, composition_strong_stress_df, "composition")
stacked_composition

,model,feature_set,test_set,mae_macro,rmse_macro,r2_macro,kl,stress_light_mae_macro,stress_light_rmse_macro,stress_light_r2_macro,stress_light_kl,stress_strong_mae_macro,stress_strong_rmse_macro,stress_strong_r2_macro,stress_strong_kl
0,xgb,extended,spatial,0.005176,0.016544,0.894023,0.006162,0.005621,0.016676,0.892709,0.006512,0.012614,0.022678,0.891443,0.022804
1,mlp,extended,spatial,0.006062,0.015942,0.932048,0.010971,0.006104,0.015971,0.932027,0.011341,0.009201,0.018333,0.928769,0.019557
2,lgbm,extended,spatial,0.005848,0.021841,0.865144,0.006184,0.006164,0.021833,0.865659,0.006833,0.011114,0.025436,0.860776,0.017305
3,elastic,extended,spatial,0.006393,0.016615,0.890858,0.010413,0.006539,0.016673,0.890868,0.010692,0.011344,0.021195,0.885361,0.027587
4,rf,extended,spatial,0.006455,0.023571,0.821228,0.009483,0.006751,0.023521,0.821461,0.009995,0.012665,0.027905,0.803526,0.021917
5,catboost,extended,spatial,0.005218,0.015467,0.930111,0.006251,0.005597,0.015588,0.929032,0.006802,0.012646,0.022212,0.912737,0.023287
6,knn,extended,spatial,0.014152,0.033337,0.871977,0.045891,0.014180,0.033282,0.872844,0.046271,0.014629,0.033702,0.871956,0.045918
7,dt,extended,spatial,0.007813,0.028853,0.679432,0.045243,0.008241,0.028803,0.680702,0.044656,0.013960,0.033298,0.656894,0.058611


In [40]:
stacked_change = merge_stress_results(change_df, change_light_stress_df, change_strong_stress_df, "change")
stacked_change

,model,feature_set,test_set,mcr,fcr,direction_accuracy,stress_light_mcr,stress_light_fcr,stress_light_direction_accuracy,stress_strong_mcr,stress_strong_fcr,stress_strong_direction_accuracy
0,xgb,extended,spatial,0.927888,1.116051,0.275925,0.929698,1.309382,0.273000,0.924011,2.205996,0.258220
1,mlp,extended,spatial,0.929698,2.248902,0.267626,0.928664,2.249418,0.268653,0.928664,2.408116,0.262172
2,lgbm,extended,spatial,0.881106,0.946498,0.270234,0.875937,0.953993,0.271182,0.869217,1.422073,0.267705
3,elastic,extended,spatial,0.921168,2.147842,0.272131,0.921944,2.144999,0.272210,0.919359,2.179633,0.264543
4,rf,extended,spatial,0.917550,1.272422,0.278612,0.920393,1.453347,0.277347,0.931248,2.289222,0.272210
5,catboost,extended,spatial,0.912380,1.110106,0.275767,0.910571,1.196692,0.274423,0.895839,1.893513,0.265650
6,knn,extended,spatial,0.772810,1.233911,0.677521,0.773843,1.237012,0.675703,0.774619,1.253037,0.675862
7,dt,extended,spatial,0.899457,1.145516,0.665349,0.891186,1.486948,0.553667,0.884983,1.957095,0.469333


In [51]:
composition_df

,model,feature_set,test_set,mae_macro,rmse_macro,r2_macro,kl
0,xgb,extended,spatial,0.005176,0.016544,0.894023,0.006162
1,mlp,extended,spatial,0.006062,0.015942,0.932048,0.010971
2,lgbm,extended,spatial,0.005848,0.021841,0.865144,0.006184
3,elastic,extended,spatial,0.006393,0.016615,0.890858,0.010413
4,rf,extended,spatial,0.006455,0.023571,0.821228,0.009483
5,catboost,extended,spatial,0.005218,0.015467,0.930111,0.006251
6,knn,extended,spatial,0.014152,0.033337,0.871977,0.045891
7,dt,extended,spatial,0.007813,0.028853,0.679432,0.045243


In [42]:
composition_light_stress_df

,model,feature_set,test_set,mae_macro,rmse_macro,r2_macro,kl
0,xgb,extended,spatial,0.005621,0.016676,0.892709,0.006512
1,mlp,extended,spatial,0.006104,0.015971,0.932027,0.011341
2,lgbm,extended,spatial,0.006164,0.021833,0.865659,0.006833
3,elastic,extended,spatial,0.006539,0.016673,0.890868,0.010692
4,rf,extended,spatial,0.006751,0.023521,0.821461,0.009995
5,catboost,extended,spatial,0.005597,0.015588,0.929032,0.006802
6,knn,extended,spatial,0.014180,0.033282,0.872844,0.046271
7,dt,extended,spatial,0.008241,0.028803,0.680702,0.044656


In [43]:
composition_strong_stress_df

,model,feature_set,test_set,mae_macro,rmse_macro,r2_macro,kl
0,xgb,extended,spatial,0.012614,0.022678,0.891443,0.022804
1,mlp,extended,spatial,0.009201,0.018333,0.928769,0.019557
2,lgbm,extended,spatial,0.011114,0.025436,0.860776,0.017305
3,elastic,extended,spatial,0.011344,0.021195,0.885361,0.027587
4,rf,extended,spatial,0.012665,0.027905,0.803526,0.021917
5,catboost,extended,spatial,0.012646,0.022212,0.912737,0.023287
6,knn,extended,spatial,0.014629,0.033702,0.871956,0.045918
7,dt,extended,spatial,0.013960,0.033298,0.656894,0.058611


In [44]:
change_df

,model,feature_set,test_set,mcr,fcr,direction_accuracy
0,xgb,extended,spatial,0.927888,1.116051,0.275925
1,mlp,extended,spatial,0.929698,2.248902,0.267626
2,lgbm,extended,spatial,0.881106,0.946498,0.270234
3,elastic,extended,spatial,0.921168,2.147842,0.272131
4,rf,extended,spatial,0.917550,1.272422,0.278612
5,catboost,extended,spatial,0.912380,1.110106,0.275767
6,knn,extended,spatial,0.772810,1.233911,0.677521
7,dt,extended,spatial,0.899457,1.145516,0.665349


In [45]:
change_light_stress_df

,model,feature_set,test_set,mcr,fcr,direction_accuracy
0,xgb,extended,spatial,0.929698,1.309382,0.273000
1,mlp,extended,spatial,0.928664,2.249418,0.268653
2,lgbm,extended,spatial,0.875937,0.953993,0.271182
3,elastic,extended,spatial,0.921944,2.144999,0.272210
4,rf,extended,spatial,0.920393,1.453347,0.277347
5,catboost,extended,spatial,0.910571,1.196692,0.274423
6,knn,extended,spatial,0.773843,1.237012,0.675703
7,dt,extended,spatial,0.891186,1.486948,0.553667


In [46]:
change_strong_stress_df

,model,feature_set,test_set,mcr,fcr,direction_accuracy
0,xgb,extended,spatial,0.924011,2.205996,0.258220
1,mlp,extended,spatial,0.928664,2.408116,0.262172
2,lgbm,extended,spatial,0.869217,1.422073,0.267705
3,elastic,extended,spatial,0.919359,2.179633,0.264543
4,rf,extended,spatial,0.931248,2.289222,0.272210
5,catboost,extended,spatial,0.895839,1.893513,0.265650
6,knn,extended,spatial,0.774619,1.253037,0.675862
7,dt,extended,spatial,0.884983,1.957095,0.469333


In [52]:
def save_dfs(dfs: dict, dir_name: str, root_dir=".."):
    """
    Save multiple DataFrames to CSV files.

    Args:
        dfs (dict): {"name": dataframe}
        dir_name (str): folder name to save into
        root_dir (str): project root (default=".")
    """

    # --- create full path ---
    save_dir = os.path.join(root_dir, dir_name)
    os.makedirs(save_dir, exist_ok=True)

    # --- save each df ---
    for name, df in dfs.items():
        file_path = os.path.join(save_dir, f"{name}.csv")
        df.to_csv(file_path, index=False)
        print(f"[SAVED] {file_path}")

In [53]:
df_dict = {"composition_df": composition_df, 
           "composition_light_stress_df": composition_light_stress_df, 
           "composition_strong_stress_df": composition_strong_stress_df, 
           "change_df": change_df, 
           "change_light_stress_df": change_light_stress_df, 
           "change_strong_stress_df": change_strong_stress_df}

save_dfs(df_dict, "cleaned_results")    

[SAVED] ../cleaned_results/composition_df.csv
[SAVED] ../cleaned_results/composition_light_stress_df.csv
[SAVED] ../cleaned_results/composition_strong_stress_df.csv
[SAVED] ../cleaned_results/change_df.csv
[SAVED] ../cleaned_results/change_light_stress_df.csv
[SAVED] ../cleaned_results/change_strong_stress_df.csv


In [54]:
temp_df = pd.read_parquet("../datasets/df_to_inference.parquet")

In [55]:
temp_df

,centroid_x,centroid_y,area_m2,curr_label__built_up,curr_label__vegetation,curr_label__water,curr_label__other,class_sum,B02_median,B02_p25,...,ndbi_median,ndbi_p25,ndbi_p75,ndbi_std,swir_used,next_built_up,next_vegetation,next_water,next_other,grid_id
0,644262.550791,5.487783e+06,62500.0,0.173214,0.798214,0.000000,0.028571,1.0,3396.0,3370.0,...,-0.174205,-0.180927,-0.169373,0.010680,1.0,0.164286,0.835714,0.000000,0.000000,4
1,644262.550791,5.488033e+06,62500.0,0.227395,0.669651,0.000000,0.102954,1.0,3344.0,3320.0,...,-0.168745,-0.173715,-0.165085,0.006269,1.0,0.174575,0.825425,0.000000,0.000000,5
2,644262.550791,5.488283e+06,62500.0,0.168157,0.830948,0.000000,0.000894,1.0,3312.0,3292.0,...,-0.169883,-0.174720,-0.165021,0.007993,1.0,0.197674,0.802326,0.000000,0.000000,6
3,644512.550791,5.478033e+06,62500.0,0.729148,0.260090,0.000000,0.010762,1.0,3622.0,3566.0,...,-0.178099,-0.182506,-0.173727,0.007379,1.0,0.730942,0.269058,0.000000,0.000000,16
4,644512.550791,5.479283e+06,62500.0,0.465095,0.275612,0.202176,0.057117,1.0,3728.0,3674.5,...,-0.175654,-0.181142,-0.169411,0.008605,1.0,0.464189,0.323663,0.194923,0.017226,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16557,687512.550791,5.484783e+06,62500.0,0.000000,1.000000,0.000000,0.000000,1.0,4252.0,4176.0,...,-0.216274,-0.224327,-0.211134,0.009930,1.0,0.000000,1.000000,0.000000,0.000000,16532
16558,687762.550791,5.482283e+06,62500.0,0.000000,1.000000,0.000000,0.000000,1.0,4584.0,4548.0,...,-0.230730,-0.234169,-0.227456,0.005034,1.0,0.000000,1.000000,0.000000,0.000000,16543
16559,687762.550791,5.484033e+06,62500.0,0.015179,0.984821,0.000000,0.000000,1.0,4664.0,4568.0,...,-0.263465,-0.278543,-0.251173,0.016399,1.0,0.033036,0.966964,0.000000,0.000000,16549
16560,687762.550791,5.484283e+06,62500.0,0.007201,0.992799,0.000000,0.000000,1.0,4484.0,4368.0,...,-0.233758,-0.243216,-0.220268,0.015548,1.0,0.007201,0.992799,0.000000,0.000000,16550


In [56]:
temp_df.columns

Index(['centroid_x', 'centroid_y', 'area_m2', 'curr_label__built_up',
       'curr_label__vegetation', 'curr_label__water', 'curr_label__other',
       'class_sum', 'B02_median', 'B02_p25', 'B02_p75', 'B02_std',
       'B03_median', 'B03_p25', 'B03_p75', 'B03_std', 'B04_median', 'B04_p25',
       'B04_p75', 'B04_std', 'B08_median', 'B08_p25', 'B08_p75', 'B08_std',
       'B11_median', 'B11_p25', 'B11_p75', 'B11_std', 'B12_median', 'B12_p25',
       'B12_p75', 'B12_std', 'ndvi_median', 'ndvi_p25', 'ndvi_p75', 'ndvi_std',
       'ndwi_median', 'ndwi_p25', 'ndwi_p75', 'ndwi_std', 'ndbi_median',
       'ndbi_p25', 'ndbi_p75', 'ndbi_std', 'swir_used', 'next_built_up',
       'next_vegetation', 'next_water', 'next_other', 'grid_id'],
      dtype='str')